# ✅ Solusi — Forecasting Mini dari Nol (Bab 7)> **Buka HANYA SETELAH** selesai mencoba (atau mentok). Sama seperti kunci kuis:> membaca solusi sebelum berjuang = ilusi kompetensi.Notebook ini mem-patch `ts_mini` dengan implementasi referensi(`solusi/ts_mini_ref.py`) lalu menjalankan alur eksperimen yang sama denganstarter — jadi semua angka bisa direproduksi. Source tiap fungsi kunciditampilkan dengan `inspect.getsource`.

## Setup (patch implementasi referensi)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'solusi'))

import inspect
import matplotlib.pyplot as plt
import numpy as np

import ts_mini_ref as ref
import ts_mini.ar as ar_mod
import ts_mini.baseline as base_mod
import ts_mini.holtwinters as hw_mod
import ts_mini.windowing as win_mod

for mod, names in [(win_mod, ['buat_window', 'split_waktu', 'buat_window_multi']),
                   (base_mod, ['mae', 'rmse', 'smape', 'forecast_naive',
                               'forecast_seasonal_naive', 'forecast_ham']),
                   (ar_mod, ['fit_ar', 'ar_forecast']),
                   (hw_mod, ['holt_winters_add', 'holt_winters_mult'])]:
    for nama in names:
        setattr(mod, nama, getattr(ref, nama))
print('ts_mini di-patch dengan implementasi referensi.')


## Kunci Implementasi### AR — konvensi lag & forecast rekursif

In [ ]:
print(inspect.getsource(ref.fit_ar))
print(inspect.getsource(ref.ar_forecast))


**Poin kunci:** baris fitur = `[y[i+p-1], ..., y[i]]` (lag-1 dulu) sehingga`coef[0]` mengalikan nilai TERBARU; di forecast, histori dibalik dulu(`hist[0]` = terbaru), prediksi masuk ke depan antrean — error menumpuk dari sini.

### Holt-Winters — urutan update level → trend → season

In [ ]:
print(inspect.getsource(ref.holt_winters_add))
print(inspect.getsource(ref.holt_winters_mult))


**Poin kunci:** loop mulai `i = m` (satu siklus penuh untuk inisialisasi);`lv_old` diambil SEBELUM level diperbarui; multiplikatif pakai `eps = 1e-8` disemua pembagi; forecast memakai `seas[n - m + (j % m)]` — musiman terakhir diulang.

## Jalankan Alur Eksperimen Lengkap (angka solusi)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np

from ts_mini.ar import ar_forecast, fit_ar
from ts_mini.baseline import (forecast_ham, forecast_naive, forecast_seasonal_naive,
                              mae, rmse, smape)
from ts_mini.data import HORIZON, SPLIT_IDX, make_mult_series, make_series
from ts_mini.holtwinters import holt_winters_add, holt_winters_mult
from ts_mini.windowing import buat_window, buat_window_multi, split_waktu

seri, t = make_series()            # aditif: trend + musiman + noise
seri_mult, _ = make_mult_series()  # multiplikatif: amplitudo ikut level
train, val = seri[:SPLIT_IDX], seri[SPLIT_IDX:]
print(f'aditif n={len(seri)} | mult n={len(seri_mult)} | train={len(train)} val={len(val)}')


In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(t, seri, alpha=0.6, label='aditif')
plt.plot(t, seri_mult, alpha=0.6, label='multiplikatif')
plt.xlabel('hari'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Dua seri terkunci — resepnya diketahui, model harus menemukannya')
plt.show()

print('contoh 5 nilai aditif:', np.round(seri[:5], 2))
print('amplitudo mult naik dengan level -> perhatikan puncak tahun ke-3 vs tahun ke-1')


In [ ]:
# Windowing: 1 seri -> dataset supervised (fungsi yang lolos test suite)
Xw, yw = buat_window(seri, 35)
Xtr, ytr, Xva, yva = split_waktu(Xw, yw, 0.8)
print(f'X={Xw.shape}, y={yw.shape} | train={len(Xtr)}, val={len(Xva)}')
assert Xw.shape == (len(seri) - 35, 35)
assert np.array_equal(Xtr, Xw[:int(len(Xw) * 0.8)]), 'split harus berurutan waktu'

# multi-step: window 35 -> prediksi 14 hari sekaligus
Xm, ym = buat_window_multi(seri, 35, 14)
print(f'multi: X={Xm.shape}, y={ym.shape}')

# visualisasi satu window + target
i0 = 100
plt.figure(figsize=(12, 4))
plt.plot(range(i0, i0 + 35), Xw[i0], 'b-', label='window (fitur)')
plt.plot([i0 + 35], [yw[i0]], 'r*', ms=15, label='target')
plt.xlabel('hari'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Satu sampel: 35 hari ke belakang -> 1 hari ke depan')
plt.show()


In [ ]:
# Tiga baseline di val 219 hari — angka yang harus dikalahkan
pred_naive = forecast_naive(train, HORIZON)
pred_sn = forecast_seasonal_naive(train, HORIZON, 365)
pred_ham = forecast_ham(train, HORIZON)

mae_naive = mae(val, pred_naive)
mae_sn = mae(val, pred_sn)
mae_ham = mae(val, pred_ham)
print(f'naive          : MAE {mae_naive:.3f} | RMSE {rmse(val, pred_naive):.3f}')
print(f'seas-naive-365 : MAE {mae_sn:.3f} | sMAPE {smape(val, pred_sn):.4f}')
print(f'HAM            : MAE {mae_ham:.3f}')
assert 8.0 < mae_naive < 10.5
assert 17.0 < mae_sn < 21.0
assert 14.0 < mae_ham < 18.0
print('✅ Baseline tercatat. Model manapun yang tidak mengalahkan ini = tidak layak.')

plt.figure(figsize=(14, 4))
plt.plot(val, 'k-', alpha=0.6, label='aktual')
plt.plot(pred_naive, 'b--', label=f'naive ({mae_naive:.1f})')
plt.plot(pred_sn, 'g--', label=f'seas-naive ({mae_sn:.1f})')
plt.xlabel('hari ke-depan'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Baseline di horizon panjang')
plt.show()


In [ ]:
# AR(7): bagus di horizon pendek, kalah di horizon panjang
coef7 = fit_ar(train, 7)
print('coef AR(7):', np.round(coef7, 4).tolist())

fc7_pendek = ar_forecast(coef7, train[-7:], 7)
akt7 = seri[SPLIT_IDX:SPLIT_IDX + 7]
ape7 = float(np.mean(np.abs(fc7_pendek - akt7) / akt7))

fc7 = ar_forecast(coef7, train[-7:], HORIZON)
mae_ar7 = mae(val, fc7)
print(f'AR(7) 7-langkah  : APE = {ape7:.4f}  (bagus!)')
print(f'AR(7) {HORIZON}-langkah : MAE = {mae_ar7:.3f}  (kalah dari naive {mae_naive:.3f}!)')
assert ape7 < 0.05
assert mae_ar7 > mae_naive, 'error rekursif harus menumpuk melebihi naive'
print('✅ Pelajaran: model bagus di 1-step ≠ bagus di multi-step. Horizon selalu disebut.')

plt.figure(figsize=(14, 4))
plt.plot(val, 'k-', alpha=0.5, label='aktual')
plt.plot(fc7, 'r-', label=f'AR(7) rekursif (MAE {mae_ar7:.1f})')
plt.xlabel('hari ke-depan'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Multi-step rekursif: prediksi kehilangan pola seiring horizon')
plt.show()


In [ ]:
# Holt-Winters di seri ADITIF
fc_add, lv, tb, ss = holt_winters_add(train, 365, 0.3, 0.05, 0.3, HORIZON)
fc_mul = holt_winters_mult(train, 365, 0.3, 0.05, 0.3, HORIZON)
mae_add = mae(val, fc_add)
mae_mul = mae(val, fc_mul)
print(f'[aditif]      level={lv:.2f} trend={tb:.4f}')
print(f'HW add  MAE: {mae_add:.3f} | HW mult MAE: {mae_mul:.3f}')
assert 20.7 < mae_add < 21.3 and 16.1 < mae_mul < 16.7

# Holt-Winters di seri MULTIPLIKATIF — di sinilah pilihan versi menentukan
tr_m, va_m = seri_mult[:SPLIT_IDX], seri_mult[SPLIT_IDX:]
fc_add_m = holt_winters_add(tr_m, 365, 0.3, 0.05, 0.3, HORIZON)[0]
fc_mul_m = holt_winters_mult(tr_m, 365, 0.3, 0.05, 0.3, HORIZON)
mae_add_m = mae(va_m, fc_add_m)
mae_mul_m = mae(va_m, fc_mul_m)
print(f'[multiplikatif] HW add MAE: {mae_add_m:.3f} | HW mult MAE: {mae_mul_m:.3f}')
assert mae_mul_m < mae_add_m, 'di seri multiplikatif, mult harus menang'
print('✅ Dua arah terbukti: pilih versi dari BENTUK amplitudo, bukan selera.')

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
axes[0].plot(va_m, 'k-', alpha=0.5, label='aktual')
axes[0].plot(fc_add_m, 'b--', label=f'add ({mae_add_m:.1f})')
axes[0].plot(fc_mul_m, 'r-', label=f'mult ({mae_mul_m:.1f})')
axes[0].set_title('Seri multiplikatif'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(val, 'k-', alpha=0.5, label='aktual')
axes[1].plot(fc_add, 'b--', label=f'add ({mae_add:.1f})')
axes[1].plot(fc_mul, 'r-', label=f'mult ({mae_mul:.1f})')
axes[1].set_title('Seri aditif'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Rolling backtest: 3 fold x 90 hari — angka per-fold, bukan cuma mean
def rolling_backtest(tr, h, build_forecast, n_folds=3):
    metrik = []
    n = len(tr)
    for k in range(n_folds):
        cut = n - (k + 1) * h
        trk, tek = tr[:cut], tr[cut:cut + h]
        fc = build_forecast(trk, h)
        metrik.append((mae(tek, fc), rmse(tek, fc)))
    return metrik

def buat_naive(trk, h):
    return forecast_naive(trk, h)

def buat_sn365(trk, h):
    return forecast_seasonal_naive(trk, h, 365)

def buat_hw(trk, h):
    return holt_winters_add(trk, 365, 0.3, 0.05, 0.3, h)[0]

hasil = {}
for nama_, fn in [('naive', buat_naive), ('seas-naive365', buat_sn365), ('HW-add', buat_hw)]:
    hasil[nama_] = rolling_backtest(train, 90, fn, 3)
    print(f'{nama_:14s}: {[(round(a, 2), round(b, 2)) for a, b in hasil[nama_]]}')
mean_naive = float(np.mean([a for a, _ in hasil['naive']]))
mean_sn = float(np.mean([a for a, _ in hasil['seas-naive365']]))
mean_hw = float(np.mean([a for a, _ in hasil['HW-add']]))
assert 3.0 < mean_naive < 25.0 and 17.0 < mean_sn < 19.0 and 14.0 < mean_hw < 19.0
assert mean_hw < mean_sn, 'HW-add konsisten menang dari seasonal-naive'
print(f'✅ mean MAE: naive {mean_naive:.2f} | seas-naive {mean_sn:.2f} | HW {mean_hw:.2f}')
print('   Perhatikan naive fold-2 (MAE ~21 vs ~5): satu fold di lembah musiman')
print('   membalikkan peringkat. Laporkan SEBARAN fold, bukan hanya mean.')


In [ ]:
print('=' * 52)
print('RINGKASAN ANGKA AKHIR (isi RUBRIK.md dari sini)')
print('=' * 52)
print(f'1. baseline val MAE       : naive {mae_naive:.2f} | seas-naive {mae_sn:.2f} | HAM {mae_ham:.2f}')
print(f'2. AR(7)                  : APE 7-step {ape7:.4f} | MAE {HORIZON}-step {mae_ar7:.2f}')
print(f'3. HW aditif / mult       : {mae_add:.2f} / {mae_mul:.2f} (seri aditif)')
print(f'4. HW di seri multipl.    : add {mae_add_m:.2f} vs mult {mae_mul_m:.2f}')
print(f'5. backtest mean MAE      : naive {mean_naive:.2f} | sn {mean_sn:.2f} | HW {mean_hw:.2f}')
print('Test/val disentuh sekali — angka ini final. Tidak ada "coba-coba lagi".')


## Jawaban Pertanyaan Analisis (bandingkan dengan versimu)

**1. Mekanisme leakage split acak.** Dengan split acak, tetangga terdekat sebuahwindow val adalah window train yang HAMPIR BERDETEKAN dengannya di waktu nyata(hari 849 dekat hari 848). Model 1-NN "menghafal" jawaban yang sudah pernahdilihat; di produksi (prediksi masa depan sungguhan) tetangga itu tidak ada.Angka yang lebih rendah mengukur kemampuan menghafal, bukan meramal. Gejalasaudaranya: in-sample pdq 6.41 vs out-of-sample 11.76 (lab Bagian 6).**2. Error rekursif menumpuk.** 7 langkah pertama masih "dikoreksi" nilai asliterakhir di histori (APE 1.9%). Setelah itu prediksi jadi input: bias arahtrend/musiman terakumulasi, forecast kehilangan pola (lihat plot AR vs aktual),MAE melampaui naive. Klaim "model bagus" tanpa menyebut horizon = iklan palsu;selalu laporkan "bagus pada h = ?".**3. AR vs musiman panjang.** AR(365) butuh 365 lag → 365 parameter + kehilangan365 baris data, dan OLS jadi tidak stabil. Trend+dummies menyelesaikan musimandengan 365 dummy + 2 kolom [1, t]: fit mudah, tapi 365 parameter mengunci noisetrain — in-sample 6.41 vs out-of-sample 11.76. Biayanya: overfit parametermusiman; solusi modern (HW) meramal musiman dengan m angka smoothing saja.**4. Memutuskan aditif vs multiplikatif.** Di eksperimen kita mult menang diKEDUA seri — tapi karena alasan berbeda: di seri aditif (21.0 vs 16.4) selisihnyadatang dari redaman drift trend oleh rasio, di seri multiplikatif (16.5 vs 12.9)dari bentuk amplitudo yang memang ikut level. Tanpa resep: (a) plot seri —amplitudo membesar saat level naik → multiplikatif (atau log1p dulu);(b) paling aman: backtest KEDUA versi di train folds, pilih mean MAE terendah.Keputusan dari data, bukan dari teks buku.**5. Mean backtest menipu.** Naive mean 10.33 "terbaik", tapi fold-2-nya 20.98 —fold itu jatuh di lembah musiman sehingga garis datar meleset jauh. Artinyanaive rapuh pada fase siklus; HW stabil antar fold (11.2/23.3/14.9 — variasilebih kecil relatif terhadap level kesulitan fold). Laporkan mean + max + sebaran(atau median), dan pilih model yang TIDAK ada fold-nya yang meledak.**6. Drift & MLOps.** Pipeline yang dijalankan ulang berkala: rolling backtestdengan data terbaru (angka: mean MAE per fold), monitoring error live (MAErolling produksi vs backtest — menyimpang jauh = drift), dan pemicu retraining.Data/seed/split dikunci ulang setiap evaluasi supaya angka antar periodesebanding (prinsip kunci data di cheatsheet §9).

---Selanjutnya: isi `RUBRIK.md` dengan angka-mu sendiri (bukan angka notebook ini),lalu lanjut ke Bab 8 (ML Produksi).